# HW04 — Data Acquisition & Ingestion


API pull (with optional API key) **and** a simple, permitted table scrape. Saves raw CSVs.


In [ ]:
import os, sys, time
from pathlib import Path
import pandas as pd

sys.path.append("src")
from config import load_env, get_key
load_env()

DATA_DIR_RAW = Path(os.getenv("DATA_DIR_RAW", "data/raw"))
DATA_DIR_RAW.mkdir(parents=True, exist_ok=True)

# ---- API Pull (example using yfinance fallback) ----
ticker = os.getenv("TICKER", "AAPL")
api_key = get_key("API_KEY", default=None)
print("Using ticker:", ticker)

def api_pull_yfinance(ticker: str) -> pd.DataFrame:
    try:
        import yfinance as yf
        hist = yf.download(ticker, period="1mo", interval="1d", progress=False)
        hist.reset_index(inplace=True)
        return hist
    except Exception as e:
        raise RuntimeError("yfinance failed; install or set a different API path.") from e

df_api = api_pull_yfinance(ticker)
assert not df_api.empty, "API dataframe is empty"
print("API columns:", list(df_api.columns))

# Basic validation
required_cols = {"Date", "Close"}
missing = required_cols - set(df_api.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

na_counts = df_api.isna().sum().sum()
print("Total NA in API df:", na_counts)

# Save
ts = __import__("datetime").datetime.now().strftime("%Y%m%d-%H%M")
api_path = DATA_DIR_RAW / f"api_yfinance_{ticker}_{ts}.csv"
df_api.to_csv(api_path, index=False)
print("Saved:", api_path)

# ---- Scrape a small table (example: Wikipedia S&P 100 list) ----
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/S%26P_100"
html = requests.get(url, timeout=20).text
soup = BeautifulSoup(html, "lxml")
table = soup.find("table", {"class": "wikitable"})
rows = []
for tr in table.find_all("tr")[1:]:
    tds = [td.get_text(strip=True) for td in tr.find_all(["td","th"])]
    if len(tds) >= 2:
        rows.append({"Symbol": tds[0], "Company": tds[1]})
df_scrape = pd.DataFrame(rows)
print("Scraped rows:", len(df_scrape))

# Validate
assert {"Symbol","Company"}.issubset(df_scrape.columns)
assert df_scrape["Symbol"].str.len().gt(0).all()

scrape_path = DATA_DIR_RAW / f"scrape_wikipedia_sp100_{ts}.csv"
df_scrape.to_csv(scrape_path, index=False)
print("Saved:", scrape_path)


Document data sources, parameters, and validation logic in this notebook.
